# Colab bootstrap

See `docs/colab.md` for the full workflow, checkpoint locations, and how to resume after a disconnect.

**Cell 6 is the only thing you edit between experiments** -- everything else stays fixed.

A version mismatch against `requirements-colab.txt`'s pins invalidates any direct comparison to the epoch-39 baseline (oil IoU 0.1057 / 0.1074 re-verified) -- see that file's header comment.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Mount Drive

Used for the protected baseline checkpoint and for persisting `output_dir` (checkpoints/metrics) across disconnects -- NOT for the training dataset itself, see step 5.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/sih26143-oil-spill-attribution'
!mkdir -p "$DRIVE_PROJECT_DIR/checkpoints" "$DRIVE_PROJECT_DIR/experiments"

## 3. Clone the repo

In [ ]:
REPO_URL = 'https://github.com/Rahil-Mokashi/sih-initial.git'
BRANCH = 'main'

!git clone -b $BRANCH $REPO_URL /content/repo
%cd /content/repo

## 4. Install pinned dependencies

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q -r requirements-colab.txt

## 5. Data + protected checkpoint

**Dataset**: downloaded directly from Zenodo (public, no auth) straight onto Colab's local disk (`/content/repo/data/`) -- NOT staged via Drive first. This is a deliberate deviation from copying a pre-built archive off Drive: the dataset is public, `scripts/download_zenodo_*.py` already exist and are used locally, and this avoids requiring a one-time ~38GB upload to Drive before anything can run. The tradeoff is a fresh download every Colab session (Colab's local disk doesn't persist) -- if that becomes a bottleneck, the next step is caching the extracted/tiled data on Drive, not yet built here (see docs/colab.md).

**Checkpoint**: the protected `checkpoints/baseline_epoch39/` copy is `checkpoints/**/*.pt`-gitignored (172MB, too large for a normal git push) -- upload it to Drive once yourself (matching `checkpoints/baseline_epoch39/MANIFEST.json`'s sha256 so you know it's the right file), then this cell copies it into the clone. Skip this cell if your experiment's config sets `resume_from: null`.

In [ ]:
# Dataset: real download, matches README.md's documented local sequence
!python scripts/download_zenodo_sample.py
!python scripts/download_zenodo_part2_parallel.py
!python scripts/download_zenodo_part2_part3.py
!python scripts/extract_zenodo_part2.py
!python scripts/extract_zenodo_part3.py
!python scripts/build_training_pool.py

In [ ]:
import os, shutil
drive_ckpt = f'{DRIVE_PROJECT_DIR}/checkpoints/baseline_epoch39/unet_resnet18_epoch39.pt'
local_ckpt_dir = '/content/repo/checkpoints/baseline_epoch39'
os.makedirs(local_ckpt_dir, exist_ok=True)
if os.path.exists(drive_ckpt):
    shutil.copy(drive_ckpt, f'{local_ckpt_dir}/unet_resnet18_epoch39.pt')
    print('copied protected baseline checkpoint from Drive')
else:
    print(f'WARNING: {drive_ckpt} not found on Drive -- upload it there first if this experiment needs resume_from')

## 6. Run the experiment

**This is the only cell you edit between experiments.** `output_dir` in the config should point under `/content/drive/...` (or be copied there after each epoch) so checkpoints/metrics survive a disconnect -- see docs/colab.md.

In [ ]:
EXPERIMENT_NAME = "baseline"
!python train.py --config configs/{EXPERIMENT_NAME}.yaml